In [1]:
from xarray_utils import analyze_netcdf, zarr_to_netcdf, find_missing_days
import pandas as pd
import xarray as xr
import numpy as np
import sklearn as sk
import sklearn as sk

In [2]:
# Open the Zarr dataset
ds_imd = xr.open_zarr("../data/raw/IMD_rainfall_0p25.zarr")
ds_imd = ds_imd.where(ds_imd != -999)

analyze_netcdf("../data/raw/IMD_rainfall_0p25.nc")

Analysis for NetCDF File: IMD_rainfall_0p25.nc

--- Dimensions ---
time: 31046
lat: 129
lon: 135

--- Coordinates ---
- lat:
    dtype: float64
    shape: (129,)
    attributes: {'axis': 'Y', 'long_name': 'latitude', 'standard_name': 'latitude', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (135,)
    attributes: {'axis': 'X', 'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- time:
    dtype: datetime64[ns]
    shape: (31046,)
    attributes: {'long_name': 'time', 'standard_name': 'time'}

--- Data Variables ---
- rain:
    dtype: float64
    shape: (31046, 129, 135)
    dimensions: ('time', 'lat', 'lon')
    attributes: {'long_name': 'Rainfall', 'units': 'mm/day'}

--- Global Attributes ---
Conventions: CF-1.7
comment: 
crs: epsg:4326
history: 2026-06-19 06:45:25.930728 Python
references: 
source: https://imdpune.gov.in/
title: IMD gridded data



In [3]:
# Open the Zarr dataset
ds_ecm = xr.open_zarr("../data/processed/s2s_reforecast_sorted.zarr")

analyze_netcdf("../data/raw/s2s_reforecast.nc")

Analysis for NetCDF File: s2s_reforecast.nc

--- Dimensions ---
time: 3720
step: 43
lat: 33
lon: 35

--- Coordinates ---
- lat:
    dtype: float64
    shape: (33,)
    attributes: {'long_name': 'latitude', 'standard_name': 'latitude', 'stored_direction': 'decreasing', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (35,)
    attributes: {'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- step:
    dtype: int64
    shape: (43,)
- time:
    dtype: datetime64[ns]
    shape: (3720,)
    attributes: {'long_name': 'initial time of forecast', 'standard_name': 'forecast_reference_time'}

--- Data Variables ---
- 10m_u_component_of_wind:
    dtype: float32
    shape: (3720, 43, 33, 35)
    dimensions: ('time', 'step', 'lat', 'lon')
    attributes: {'GRIB_NV': np.int64(0), 'GRIB_Nx': np.int64(35), 'GRIB_Ny': np.int64(33), 'GRIB_cfName': 'eastward_wind', 'GRIB_cfVarName': 'u10', 'GRIB_dataType': 'cf', 'GRIB_gridDefinitionDescription': 'Latitude/longi

In [4]:

# convert lead days into timedeltas
step_td = pd.to_timedelta(ds_ecm.step.values, unit="D").to_numpy()

ds_ecmv = ds_ecm.assign_coords(
    valid_time=(("time", "step"),
                ds_ecm.time.values[:, None] + step_td[None, :])
)

In [5]:
# ======================================================================
# S2S multi-window UNet -- LOSS-FAMILY SWEEP VERSION
# Single-cell Jupyter script. Set LOSS_NAME below, run, compare.
#
# Losses available (all masked, all anomaly-space):
#   "mse"          plain masked MSE                     -> conditional MEAN
#   "mae"          masked L1                            -> conditional MEDIAN
#   "huber"        smooth-L1, delta-tunable             -> robust mean/median blend
#   "wmse"         intensity-weighted MSE (Chandel-style extremes weighting)
#   "tail"         tail-weighted: MSE * (1 + a*|y|/std)^p
#   "pinball"      asymmetric quantile loss at one tau (median-ish, tilted)
#   "logcosh"      smooth, MAE-like tails, MSE-like centre
#   "grad"         MSE + spatial-gradient matching (structure, not amplitude)
#   "spectral"     MSE + radial power-spectrum matching (sharpness/structure)
#
# NOTE ON EXPECTATIONS: mse/mae/huber/wmse/tail/pinball/logcosh are all
# POINTWISE -- each converges to some conditional central statistic, so they
# tend to sit on the same skill/ACC frontier and mostly trade WHICH errors
# they tolerate, not how much total signal they extract. "grad" and
# "spectral" are the two that score spatial STRUCTURE rather than per-cell
# error, so they are the ones with a mechanism to move skill and ACC
# together. Run them last if the pointwise family disappoints.
# ======================================================================


# decorator+imports+Args


In [6]:

import os, time, json
from contextlib import contextmanager
import numpy as np

# ---------------- CONFIG ----------------
LOSS_NAME = "mae"          # <-- change this per run
LOSS_KW = {}               # per-loss knobs, e.g. {"delta": 1.0} for huber
                           #   huber   : delta (default 1.0)
                           #   wmse    : power (default 1.0)
                           #   tail    : alpha (default 1.0), power (default 1.0)
                           #   pinball : tau (default 0.5)
                           #   grad    : w_grad (default 1.0)
                           #   spectral: w_spec (default 1.0)

IMD_TARGET_VAR = "rain"
WINDOWS = [("week2", 8, 14), ("week3_4", 15, 28), ("week5_6", 29, 42)]
CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
MONTHS = None
TEST_YEARS_N = 3
CACHE = "../data/cache/unet_cache_mw.npz"
DTYPE = np.float32

class Args:
    cmd = "prepare"          # "prepare" or "train"
    epochs = 60
    batch = 16
    base = 24
    drop = 0.2
    wd = 1e-3
    lr = 2e-4
    patience = 10
    folds = 5

args = Args()
ARG_DICT = {k: getattr(args, k) for k in dir(args) if not k.startswith("_")}
ARG_DICT["loss"] = LOSS_NAME
ARG_DICT["loss_kw"] = LOSS_KW

# separate outputs per loss so runs never collide or resume each other
tag = LOSS_NAME + ("_" + "_".join(f"{k}{v}" for k, v in LOSS_KW.items()) if LOSS_KW else "")
OUT_MAPS = f"../results/models/unet_mw_{tag}.nc"


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)



# prepare

In [ ]:
# ======================================================================
# PREPARE
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.reshape((366,) + shp).astype(DTYPE)


def prepare(ecmwf_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    with stage("Coarse subset over padded India box"):
        ecmwf_ds = ensure_valid_time(ecmwf_ds)
        for c in ("lat", "lon"):
            if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
                ecmwf_ds = ecmwf_ds.sortby(c)
        la0, la1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
        lo0, lo1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
        sub = ecmwf_ds.sel(lat=slice(la0 - COARSE_PAD, la1 + COARSE_PAD),
                           lon=slice(lo0 - COARSE_PAD, lo1 + COARSE_PAD))
        if MONTHS is not None:
            sub = sub.sel(time=sub["time"].dt.month.isin(list(MONTHS)))
        feature_vars = list(sub.data_vars)
        clat, clon = sub["lat"].values, sub["lon"].values
        leads = (sub["step"].values / np.timedelta64(1, "D")).astype(int)
        init = sub["time"].values
        print(f"    {dict(sub.sizes)} x {len(feature_vars)} vars")

    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    X_list, y_list, doy_list, wid_list = [], [], [], []
    for wid, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi})"):
            sel = np.where((leads >= lo) & (leads <= hi))[0]
            if len(sel) == 0:
                raise ValueError(f"no leads in [{lo},{hi}]")
            with ProgressBar():
                #caluclate mean
                Xw = sub.isel(step=sel).mean(dim="step").compute()
            Xa = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)
            windows= Xa.shape[1]
            n_samples, n_vars, h, w = Xa.shape
            K = len(WINDOWS)
            X_rolling = np.zeros((n_samples, K * n_vars, h, w), dtype=np.float32)
            for i in range(n_samples):
                w_idx = wid[i]  # Current window index (0 for Week 1, 1 for Week 2, etc.)
    
    # Fill channels cumulatively up to the current window
    # Week 1 fills 0:5
    # Week 2 fills 0:10
    # Week 3 fills 0:15...
                for w in range(w_idx + 1):
                    X_rolling[i, w*n_vars : (w+1)*n_vars] = Xa[i, w]
            vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)
            centre = init + np.timedelta64((lo + hi) // 2, "D")
            doya = xr.DataArray(centre, dims="t").dt.dayofyear.values
            X_list.append(X_rolling); y_list.append(ya)
            doy_list.append(doya); wid_list.append(np.full(len(Xa), wid, dtype=np.int64))
            print(f"    +{len(Xa)} samples")

    X = np.concatenate(X_list); y = np.concatenate(y_list)
    doy = np.concatenate(doy_list); wid = np.concatenate(wid_list)
    year = np.concatenate([init.astype("datetime64[Y]").astype(int) + 1970] * len(WINDOWS))
    print(f"\n    total {len(X)} samples ({len(init)} inits x {len(WINDOWS)} windows)")

    with stage("Strict mask + test-year holdout"):
        mask = np.isfinite(y).all(axis=0)
        print(f"    strict mask: {int(mask.sum())} cells")
        uy = np.unique(year)
        test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    test years: {sorted(test_years)}")

    with stage("Caching"):
        np.savez_compressed( 
            cache_path, X=X, y=y, doy=doy, wid=wid, year=year,
            mask=mask, is_test=is_test, clat=clat, clon=clon,
            flat_lat=flat_lat, flat_lon=flat_lon,
            feature_vars=np.array(feature_vars),
            window_names=np.array([w[0] for w in WINDOWS]))
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB -> {cache_path}")


def anomalise_fold(X, y, doy, tr):
    """Train-fold-only climatology + standardisation. Leakage-critical."""
    clim_y = _clim_grid(y[tr], doy[tr], CLIM_WINDOW_DAYS)
    clim_X = _clim_grid(X[tr], doy[tr], CLIM_WINDOW_DAYS)
    ya = y - clim_y[doy - 1]
    Xa = X - clim_X[doy - 1]
    xm = np.nanmean(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.nanstd(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.where(xs < 1e-8, 1.0, xs)
    Xa = np.nan_to_num((Xa - xm) / xs)
    return Xa.astype(DTYPE), ya.astype(DTYPE), clim_y


#run the prepare dataset
prepare(ds_ecmv, ds_imd, CACHE)

[ ] Coarse subset over padded India box ...
    step is int64 (max 42) -> units='D'
    {'time': 3720, 'step': 43, 'lat': 33, 'lon': 35} x 21 vars
[x] Coarse subset over padded India box  (0.0s)
[ ] Window week2 (days 8-14) ...
[########################################] | 100% Completed | 6.44 sms


TypeError: 'int' object is not subscriptable

In [ ]:
def prepare(ecmwf_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    with stage("Coarse subset over padded India box"):
        ecmwf_ds = ensure_valid_time(ecmwf_ds)
        for c in ("lat", "lon"):
            if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
                ecmwf_ds = ecmwf_ds.sortby(c)
        la0, la1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
        lo0, lo1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
        sub = ecmwf_ds.sel(lat=slice(la0 - COARSE_PAD, la1 + COARSE_PAD),
                           lon=slice(lo0 - COARSE_PAD, lo1 + COARSE_PAD))
        if MONTHS is not None:
            sub = sub.sel(time=sub["time"].dt.month.isin(list(MONTHS)))
        feature_vars = list(sub.data_vars)
        clat, clon = sub["lat"].values, sub["lon"].values
        leads = (sub["step"].values / np.timedelta64(1, "D")).astype(int)
        init = sub["time"].values
        n_inits = len(init)
        n_win = len(WINDOWS)
        n_var = len(feature_vars)
        h, w = len(clat), len(clon)
        print(f"    {dict(sub.sizes)} x {n_var} vars")

    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    # Step 1: Collect 5D structured array: (n_inits, n_win, n_var, h, w)
    X_structured = np.empty((n_inits, n_win, n_var, h, w), dtype=DTYPE)
    y_list, doy_list, wid_list = [], [], []

    for wid, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi})"):
            sel = np.where((leads >= lo) & (leads <= hi))[0]
            if len(sel) == 0:
                raise ValueError(f"no leads in [{lo},{hi}]")
            with ProgressBar():
                Xw = sub.isel(step=sel).mean(dim="step").compute()
            
            # Store into structured window slot
            Xa = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)
            X_structured[:, wid] = Xa

            vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)
            centre = init + np.timedelta64((lo + hi) // 2, "D")
            doya = xr.DataArray(centre, dims="t").dt.dayofyear.values
            
            y_list.append(ya)
            doy_list.append(doya)
            wid_list.append(np.full(n_inits, wid, dtype=np.int64))
            print(f"    +{len(Xa)} samples")

    # Step 2: Build the Rolling / Cumulative Feature Canvas
    # Total fixed input channels = max_windows * n_vars (e.g. 4 * 5 = 20 channels)
    max_channels = n_win * n_var
    X_rolling = np.zeros((n_inits * n_win, max_channels, h, w), dtype=DTYPE)

    for i in range(n_inits):
        for w_idx in range(n_win):
            sample_idx = w_idx * n_inits + i  # Matches concatenated order
            
            # Copy all windows up to the current window w_idx
            # Window 0 (Week 1) -> fills channels 0 : n_var (rest stay 0)
            # Window 1 (Week 2) -> fills channels 0 : 2*n_var (rest stay 0)
            # Window 2 (Week 3) -> fills channels 0 : 3*n_var ...
            for past_w in range(w_idx + 1):
                c_start = past_w * n_var
                c_end = (past_w + 1) * n_var
                X_rolling[sample_idx, c_start:c_end] = X_structured[i, past_w]

    # Concatenate targets and metadata
    y = np.concatenate(y_list)
    doy = np.concatenate(doy_list)
    wid = np.concatenate(wid_list)
    year = np.concatenate([init.astype("datetime64[Y]").astype(int) + 1970] * n_win)
    
    X = X_rolling  # Now shape: (n_inits * n_win, n_win * n_var, h, w)
    print(f"\n    total {len(X)} samples ({n_inits} inits x {n_win} windows)")
    print(f"    rolling X shape: {X.shape} (Total input channels: {X.shape[1]})")

    with stage("Strict mask + test-year holdout"):
        mask = np.isfinite(y).all(axis=0)
        print(f"    strict mask: {int(mask.sum())} cells")
        uy = np.unique(year)
        test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    test years: {sorted(test_years)}")

    with stage("Caching"):
        np.savez_compressed(
            cache_path, X=X, y=y, doy=doy, wid=wid, year=year,
            mask=mask, is_test=is_test, clat=clat, clon=clon,
            flat_lat=flat_lat, flat_lon=flat_lon,
            feature_vars=np.array(feature_vars),
            window_names=np.array([w[0] for w in WINDOWS]))
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB -> {cache_path}")

#run the prepare dataset
prepare(ds_ecmv, ds_imd, CACHE)

[ ] Coarse subset over padded India box ...
    step is int64 (max 42) -> units='D'
    {'time': 3720, 'step': 43, 'lat': 33, 'lon': 35} x 21 vars
[x] Coarse subset over padded India box  (0.1s)
[ ] Window week2 (days 8-14) ...
[########################################] | 100% Completed | 7.16 sms
[########################################] | 100% Completed | 1.37 sms
    +3720 samples
[x] Window week2 (days 8-14)  (11.8s)
[ ] Window week3_4 (days 15-28) ...


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_51819/763598886.py:49: RuntimeWarning: Mean of empty slice
  ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)


[########################################] | 100% Completed | 6.95 sms
[########################################] | 100% Completed | 2.52 sms
    +3720 samples
[x] Window week3_4 (days 15-28)  (16.0s)
[ ] Window week5_6 (days 29-42) ...


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_51819/763598886.py:49: RuntimeWarning: Mean of empty slice
  ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)


[########################################] | 100% Completed | 6.24 sms
[########################################] | 100% Completed | 2.53 sms
    +3720 samples
[x] Window week5_6 (days 29-42)  (15.2s)


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_51819/763598886.py:49: RuntimeWarning: Mean of empty slice
  ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)



    total 11160 samples (3720 inits x 3 windows)
    rolling X shape: (11160, 63, 33, 35) (Total input channels: 63)
[ ] Strict mask + test-year holdout ...
    strict mask: 4964 cells
    test years: [np.int64(2022), np.int64(2023), np.int64(2024)]
[x] Strict mask + test-year holdout  (0.1s)
[ ] Caching ...
    1.99 GB -> unet_cache_mw.npz
[x] Caching  (52.0s)


In [ ]:
# ======================================================================
# LOSS FAMILY
# ======================================================================

def make_loss(name, target_std, **kw):
    """Return fn(pred, target, mask) -> scalar. All masked, all normalised
    by mask.sum() (never numel), all operating on anomalies.

    target_std: scalar float, train-set anomaly std. Used to make the
    intensity-weighted variants scale-free so their knobs mean the same
    thing regardless of units.
    """
    import torch
    import torch.nn.functional as F

    def _m(x, m):
        """mask, sum, normalise by mask -- the one correct reduction."""
        return (x * m).sum() / m.sum().clamp(min=1.0)

    if name == "mse":
        def f(p, t, m):
            return _m((p - t) ** 2, m)

    elif name == "mae":
        def f(p, t, m):
            return _m((p - t).abs(), m)

    elif name == "huber":
        delta = kw.get("delta", 1.0)
        def f(p, t, m):
            e = (p - t).abs()
            q = torch.clamp(e, max=delta)
            return _m(0.5 * q ** 2 + delta * (e - q), m)

    elif name == "logcosh":
        def f(p, t, m):
            e = p - t
            # numerically stable log(cosh(e))
            return _m(e + F.softplus(-2.0 * e) - float(np.log(2.0)), m) \
                if False else _m(torch.log(torch.cosh(torch.clamp(e, -12, 12))), m)

    elif name == "wmse":
        # intensity-weighted MSE: weight each cell by how extreme the OBS is.
        # This is the Chandel-style "weighted loss for extremes" idea.
        power = kw.get("power", 1.0)
        def f(p, t, m):
            w = 1.0 + (t.abs() / target_std) ** power
            return _m(w * (p - t) ** 2, m)

    elif name == "tail":
        # like wmse but with an explicit strength knob; alpha=0 -> plain MSE
        alpha = kw.get("alpha", 1.0)
        power = kw.get("power", 1.0)
        def f(p, t, m):
            w = (1.0 + alpha * (t.abs() / target_std)) ** power
            return _m(w * (p - t) ** 2, m)

    elif name == "pinball":
        # asymmetric: under-prediction costs tau/(1-tau) more than over.
        # tau=0.5 == MAE/2. tau>0.5 pushes predictions UP (anti-hedge for wet).
        tau = kw.get("tau", 0.5)
        def f(p, t, m):
            e = t - p
            return _m(torch.maximum(tau * e, (tau - 1.0) * e), m)

    elif name == "grad":
        # MSE + spatial-gradient matching. Scores STRUCTURE, not per-cell
        # amplitude, so it has a mechanism the pointwise family lacks.
        wg = kw.get("w_grad", 1.0)
        def f(p, t, m):
            base = _m((p - t) ** 2, m)
            mx = m[:, :, 1:] * m[:, :, :-1]
            my = m[:, 1:, :] * m[:, :-1, :]
            gx = ((p[:, :, 1:] - p[:, :, :-1]) - (t[:, :, 1:] - t[:, :, :-1])) ** 2
            gy = ((p[:, 1:, :] - p[:, :-1, :]) - (t[:, 1:, :] - t[:, :-1, :])) ** 2
            return base + wg * (_m(gx, mx) + _m(gy, my))

    elif name == "spectral":
        # MSE + radially-averaged power spectrum matching. Directly targets
        # the "deterministic nets lose power at fine scales" problem.
        ws = kw.get("w_spec", 1.0)
        def f(p, t, m):
            base = _m((p - t) ** 2, m)
            pf = torch.fft.rfft2(p * m)
            tf = torch.fft.rfft2(t * m)
            pp = (pf.real ** 2 + pf.imag ** 2 + 1e-8).mean(dim=0)
            tp = (tf.real ** 2 + tf.imag ** 2 + 1e-8).mean(dim=0)
            return base + ws * (torch.log(pp) - torch.log(tp)).abs().mean()

    else:
        raise ValueError(f"unknown LOSS_NAME '{name}'")

    return f

# model

# train

In [ ]:
# ======================================================================
# MODEL + TRAIN
# ======================================================================
args.cmd = "train"

def build_and_run(args, cache_path, out_maps, loss_name, loss_kw):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    dev = ("cuda" if torch.cuda.is_available()
           else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"    device: {dev} | loss: {loss_name} {loss_kw}")

    z = np.load(cache_path, allow_pickle=False)
    X, y, doy, wid = z["X"], z["y"], z["doy"], z["wid"]
    year, mask, is_test = z["year"], z["mask"], z["is_test"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    n_win = len(z["window_names"]); H, W = len(flat_lat), len(flat_lon)
    n_var = X.shape[1]

    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp_np = np.stack([gxx, gyy], -1).astype(np.float32)[None]
    assert np.abs(samp_np).max() <= 1.0, "fine grid outside coarse box"

    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static_np = np.stack([mask.astype(DTYPE),
                          np.broadcast_to(lat2, (H, W)).astype(DTYPE),
                          np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(self, ci, co, drop=0.0):
            super().__init__()
            L = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                 nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if drop > 0:
                L.append(nn.Dropout2d(drop))
            self.f = nn.Sequential(*L)
        def forward(self, x):
            return self.f(x)

    class MWUNet(nn.Module):
        def __init__(self, n_var, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            self.emb = nn.Embedding(n_win, emb)
            self.enc_c1 = Block(n_var + emb, base * 2)
            self.enc_c2 = Block(base * 2, base * 2)
            self.inp = Block(base * 2 + 3, base)
            self.d1 = Block(base, base * 2, drop)
            self.d2 = Block(base * 2, base * 4, drop)
            self.bott = Block(base * 4, base * 4, drop)
            self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            self.du2 = Block(base * 4, base * 2, drop)
            self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            self.du1 = Block(base * 2, base)
            self.head = nn.Conv2d(base, 1, 1)
            self.pool = nn.MaxPool2d(2)
        def forward(self, xc, wid, samp, static):
            b, _, h, w = xc.shape
            e = self.emb(wid)[:, :, None, None].expand(-1, -1, h, w)
            c = self.enc_c2(self.enc_c1(torch.cat([xc, e], 1)))
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1),
                              mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], 1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = self.inp(f); e1 = self.d1(self.pool(e0)); e2 = self.d2(self.pool(e1))
            u = self.du2(torch.cat([self.u2(self.bott(e2)), e1], 1))
            u = self.du1(torch.cat([self.u1(u), e0], 1))
            return self.head(u)[:, :, :H0, :W0].squeeze(1)

    samp = torch.tensor(samp_np).to(dev)
    stat = torch.tensor(static_np).to(dev)

    def skill_acc(p, t, fin):
        se_m = np.where(fin, (t - p) ** 2, np.nan)
        se_c = np.where(fin, t ** 2, np.nan)
        with np.errstate(invalid="ignore"):
            rmse_m = np.sqrt(np.namean(se_m, axis=0))
            rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
            ok = rmse_c > 1e-6
            skill = np.where(ok, 1 - rmse_m / np.where(ok, rmse_c, 1), np.nan)
            tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
            pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
            num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
            den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                          * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
            acc = np.where(den > 0, num / den, np.nan)
        return skill, acc

    def std_ratio(p, t, fin):
        """Diagnostic only -- reported, never optimised."""
        pv, tv = p[fin], t[fin]
        return float(pv.std() / tv.std()) if tv.std() > 0 else np.nan

    def train_one(tr_idx, va_idx, Xa, ya, max_epochs):
        Xt = torch.tensor(Xa)
        yt = torch.tensor(np.nan_to_num(ya))
        widt = torch.tensor(wid)
        fin = torch.tensor((np.isfinite(ya) & mask[None]).astype(DTYPE))

        tstd = float(np.nanstd(ya[tr_idx][np.isfinite(ya[tr_idx])]))
        crit = make_loss(loss_name, tstd, **loss_kw)

        def dl(idx, sh):
            return DataLoader(TensorDataset(Xt[idx], widt[idx], yt[idx], fin[idx]),
                              batch_size=args.batch, shuffle=sh)

        tr_dl, va_dl = dl(tr_idx, True), dl(va_idx, False)
        model = MWUNet(n_var, n_win, base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            model.train()
            for xb, wb, yb, mb in tr_dl:
                xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                loss = crit(model(xb, wb, samp, stat), yb, mb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            vl, n = 0.0, 0
            with torch.no_grad():
                for xb, wb, yb, mb in va_dl:
                    xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                    vl += float(crit(model(xb, wb, samp, stat), yb, mb)) * len(xb)
                    n += len(xb)
            vl /= n
            if vl < best - 1e-6:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                wait += 1
            if wait >= args.patience:
                break
        model.load_state_dict(best_state)
        return model, best, ep + 1

    def predict(model, idx, Xa):
        Xt = torch.tensor(Xa); widt = torch.tensor(wid)
        out = []
        model.eval()
        with torch.no_grad():
            for a in range(0, len(idx), args.batch):
                j = idx[a:a + args.batch]
                out.append(model(Xt[j].to(dev), widt[j].to(dev), samp, stat).cpu().numpy())
        return np.concatenate(out)

    # ---------- rotating-year CV ----------
    nontest_years = sorted(set(year[~is_test].tolist()))
    blocks = np.array_split(nontest_years, args.folds)
    resume_path = out_maps.replace(".nc", "_folds.json")
    done = {}
    if os.path.exists(resume_path):
        with open(resume_path) as fh:
            done = {int(k): v for k, v in json.load(fh).items()}
        print(f"    resuming: folds {sorted(done)} cached ({resume_path})")

    with stage(f"Rotating-year CV: {args.folds} folds over {len(nontest_years)} years"):
        for fi, val_years in enumerate(blocks):
            if fi in done:
                r = done[fi]
                print(f"    fold {fi} (cached): skill {r['skill']:+.3f} | ACC {r['acc']:.3f}")
                continue
            val_years = set(val_years.tolist())
            va_i = np.isin(year, list(val_years)) & ~is_test
            tr_i = ~np.isin(year, list(val_years)) & ~is_test
            Xa, ya, _ = anomalise_fold(X, y, doy, tr_i)
            model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0],
                                          Xa, ya, args.epochs)
            p = predict(model, np.where(va_i)[0], Xa)
            t = ya[va_i]
            fin = np.isfinite(y[va_i]) & mask[None]
            sk, ac = skill_acc(p, t, fin)
            ms, ma = float(np.nanmean(sk[mask])), float(np.nanmean(ac[mask]))
            sr = std_ratio(p, t, fin)
            done[fi] = {"skill": ms, "acc": ma, "std_ratio": sr,
                        "val_years": sorted(val_years), "epochs": int(eps),
                        "vloss": float(vloss)}
            with open(resume_path, "w") as fh:
                json.dump({str(k): v for k, v in done.items()}, fh, indent=2)
            print(f"    fold {fi} val {sorted(val_years)}: skill {ms:+.3f} | "
                  f"ACC {ma:.3f} | std_ratio {sr:.3f} | {eps} ep   [saved]", flush=True)

        ks = [i for i in range(args.folds) if i in done]
        fs = [done[i]["skill"] for i in ks]; fa = [done[i]["acc"] for i in ks]
        fr = [done[i].get("std_ratio", np.nan) for i in ks]
        print(f"\n    [{loss_name}]  CV skill {np.mean(fs):+.4f} +/- {np.std(fs):.4f}"
              f" | CV ACC {np.mean(fa):.4f} +/- {np.std(fa):.4f}"
              f" | std_ratio {np.nanmean(fr):.3f}")

    # ---------- final model -> test ----------
    with stage("Final model on all non-test years -> test"):
        es_years = set(nontest_years[-2:])
        va_i = np.isin(year, list(es_years)) & ~is_test
        fit_i = (~is_test) & ~va_i
        Xa, ya, _ = anomalise_fold(X, y, doy, fit_i)
        model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0], Xa, ya, args.epochs)

        te_i = np.where(is_test)[0]
        p = predict(model, te_i, Xa)
        t = ya[is_test]
        fin = np.isfinite(y[is_test]) & mask[None]

        import xarray as xr
        wid_te = wid[is_test]; data_vars = {}
        print(f"    trained {eps} ep")
        summary = {}
        for w in range(n_win):
            sm = wid_te == w
            if sm.sum() == 0:
                continue
            sk, ac = skill_acc(p[sm], t[sm], fin[sm])
            sr = std_ratio(p[sm], t[sm], fin[sm])
            wn = str(z["window_names"][w])
            data_vars[f"{wn}_skill"] = (("lat", "lon"), sk)
            data_vars[f"{wn}_acc"] = (("lat", "lon"), ac)
            summary[wn] = {"skill": float(np.nanmean(sk[mask])),
                           "acc": float(np.nanmean(ac[mask])),
                           "std_ratio": sr,
                           "pct_pos": float(100 * np.nanmean(sk[mask] > 0))}
            print(f"    test {wn:>8}: skill {summary[wn]['skill']:+.4f} | "
                  f"ACC {summary[wn]['acc']:.4f} | std_ratio {sr:.3f} | "
                  f"{summary[wn]['pct_pos']:.0f}% cells+")

        out = xr.Dataset(data_vars, coords={"lat": flat_lat, "lon": flat_lon})
        out.attrs["loss"] = f"{loss_name} {loss_kw}"
        out.attrs["windows"] = ", ".join(f"{n}:{lo}-{hi}" for n, lo, hi in WINDOWS)
        out.to_netcdf(out_maps)
        torch.save({"state": model.state_dict(), "args": ARG_DICT}, out_maps.replace(".nc", ".pt"))

        # append to a cross-loss comparison table
        tbl = "../results/models/loss_comparison.json"
        allr = json.load(open(tbl)) if os.path.exists(tbl) else {}
        allr[tag] = {"cv_skill": float(np.mean(fs)), "cv_acc": float(np.mean(fa)),
                     "cv_std_ratio": float(np.nanmean(fr)), "test": summary}
        json.dump(allr, open(tbl, "w"), indent=2)
        print(f"    -> {out_maps} (+ .pt); comparison appended to {tbl}")


# ======================================================================
if args.cmd == "prepare":
    prepare(ds_ecmv, ds_imd, CACHE)          # noqa: F821
else:
    if not os.path.exists(CACHE):
        raise SystemExit(f"run prepare first ({CACHE} missing)")
    build_and_run(args, CACHE, OUT_MAPS, LOSS_NAME, LOSS_KW)

    # print the running cross-loss comparison
    tbl = "../results/models/loss_comparison.json"
    if os.path.exists(tbl):
        allr = json.load(open(tbl))
        print(f"\n{'loss':>22} {'CV skill':>10} {'CV ACC':>8} {'std':>6} "
              f"{'w3_4 skill':>11} {'w3_4 ACC':>9}")
        for k, v in sorted(allr.items()):
            w = v["test"].get("week3_4", {})
            print(f"{k:>22} {v['cv_skill']:>+10.4f} {v['cv_acc']:>8.4f} "
                  f"{v['cv_std_ratio']:>6.3f} {w.get('skill', float('nan')):>+11.4f} "
                  f"{w.get('acc', float('nan')):>9.4f}")

    device: mps | loss: mae {}
[ ] Rotating-year CV: 5 folds over 17 years ...


KeyboardInterrupt: 